# Mini-MIAS: Attention-Driven **Weakly-Supervised Lesion Localization**

Trained on **image-level labels only** (Normal vs Abnormal, no bounding boxes), an
EfficientNetB0 backbone with an inserted **SE / CBAM attention block** learns to
*localize* the lesion. We validate localization **quantitatively** against the MIAS
ground-truth lesion coordinates in `Info1.txt` (`x y radius`).

**Pipeline:** whole mammogram -> CLAHE -> EfficientNetB0 (ImageNet, frozen to `block6a`)
-> **attention (SE / CBAM)** -> GAP -> Dense(1).  A trained-feature RBF-SVM head is also
provided (the attention features it uses are the *trained* ones, fixing the common bug of
extracting from a fresh, randomly-initialised attention block).

**Two contributions, scored together:**
1. **Classification** - Normal vs Abnormal, leakage-safe (image-wise split, train-only aug).
2. **Localization** - Grad-CAM / CBAM spatial map vs GT lesion disk: *pointing-game hit-rate*,
   *normalized centre-distance*, *attention-vs-lesion IoU*.

> Novelty angle for publication: the attention map is the scientific object (explainable /
> weakly-supervised localization), distinct from a pure classification benchmark.

**MIAS facts baked in:** images are `1024x1024`; the `Info1.txt` origin is **bottom-left**
(so `y_top = 1024 - y`); 322 images (207 Normal / 115 Abnormal); 111 abnormal images carry
lesion coordinates (7 have >1 lesion).

> **TensorFlow-GPU on Windows:** native Windows TF supports GPU only up to **TF 2.10**.
> TF 2.11+ needs **WSL2** or `tensorflow-directml-plugin`. If the GPU check prints an empty
> list, TF is on CPU (still runnable, just slower).


In [ ]:
# ---- GPU setup: PIN to the discrete NVIDIA GPU ----
# TF-DirectML enumerates BOTH the RTX 4050 and the Intel iGPU. The iGPU shares (scarce) system
# RAM and triggers fatal 'CPU->PluggableDevice Memcpy' OOM crashes. Restrict to the NVIDIA GPU.
import os
os.environ.setdefault('DML_VISIBLE_DEVICES', '0')   # RTX=adapter 0; hide Intel iGPU (must precede TF import)
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow version:', tf.__version__)
print('GPUs detected     :', gpus)
if gpus:
    pick = gpus[0]
    for g in gpus:
        try:
            nm = tf.config.experimental.get_device_details(g).get('device_name', '')
        except Exception:
            nm = ''
        if 'NVIDIA' in nm.upper() or 'RTX' in nm.upper():
            pick = g; break
    try:
        tf.config.set_visible_devices([pick], 'GPU')
        tf.config.experimental.set_memory_growth(pick, True)
        print('Pinned to:', pick)
    except Exception as e:
        print('GPU config error:', e)
else:
    print('\n[WARNING] No GPU visible -> running on CPU (slower but fine).')
print('Visible GPUs:', tf.config.get_visible_devices('GPU'))

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
# ---------------- Config (LOCAL Windows paths; forward slashes are fine) ----------------
IMAGE_DIR = 'G:/My Drive/Colab/all-mias/'
INFO_PATH = 'G:/My Drive/Colab/all-mias/Info1.txt'
SAVE_DIR  = 'G:/My Drive/VS_CC/models'
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE   = 224
ORIG_SIZE  = 1024                     # MIAS native resolution
TAP_LAYER  = 'block6a_activation'     # EfficientNetB0 tap point (~14x14 feature map)
ATTENTION  = 'cbam'                   # 'none' | 'se' | 'cbam' | 'se_cbam'
ROT_ANGLES = [0, 90, 180, 270]        # train-only augmentation
BATCH_SIZE = 16
EPOCHS     = 30
RANDOM_STATE = 2021

# MIAS Info1.txt coordinate origin is the BOTTOM-left corner -> flip y to image (top-left) coords.
MIAS_Y_FROM_BOTTOM = True

# NOTE: EfficientNet (keras.applications) has an internal Normalization layer and expects
# pixel values in [0, 255]. We therefore keep images in [0, 255] float (NO /255 divide).

## 1. Parse `Info1.txt` -> labels **+ lesion coordinates**

One row per image. `label`: `0 = Normal`, `1 = Abnormal`. For abnormal images we also keep
every lesion as `(x, y, radius)` in raw MIAS convention (converted to image coords later).
Abnormal images without coordinates (a few `ARCH`/`ASYM` rows) stay labelled abnormal but are
skipped in the localization eval.

In [ ]:
def build_label_table(info_path):
    rows = {}
    with open(info_path, 'r') as f:
        for line in f:
            p = line.split()
            if len(p) < 3 or not p[0].startswith('mdb'):
                continue
            img_id, cls = p[0], p[2]
            if img_id not in rows:
                rows[img_id] = {'image_id': img_id, 'label': 0, 'severity': 'N', 'lesions': []}
            if cls != 'NORM':
                rows[img_id]['label'] = 1
                sev = p[3] if len(p) > 3 else ''
                if sev in ('B', 'M'):
                    rows[img_id]['severity'] = sev
                if len(p) >= 7:
                    try:
                        rows[img_id]['lesions'].append((int(p[4]), int(p[5]), int(p[6])))
                    except ValueError:
                        pass
    df = pd.DataFrame(rows.values()).sort_values('image_id').reset_index(drop=True)
    return df

df = build_label_table(INFO_PATH)
print('Total unique images :', len(df))
print(df['label'].value_counts().rename({0: 'Normal', 1: 'Abnormal'}).to_string())
print('Abnormal w/ coords  :', int(((df['label'] == 1) & (df['lesions'].str.len() > 0)).sum()))
df.head()

## 2. Preprocessing + coordinate scaling

Whole image -> resize 224 -> CLAHE -> 3-channel float in **[0, 255]** (EfficientNet-ready).
`scale_lesion` maps a MIAS `(x, y, r)` at 1024 (origin bottom-left) to `(cx, cy, r)` at 224
(origin top-left).

In [ ]:
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def load_and_preprocess(img_id):
    path = os.path.join(IMAGE_DIR, img_id + '.pgm')
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = _clahe.apply(img)
    img = np.stack([img] * 3, axis=-1).astype(np.float32)   # keep [0,255] for EfficientNet
    return img

def rotate(img, angle):
    if angle == 0:
        return img
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REFLECT_101)

def scale_lesion(x, y, r):
    if MIAS_Y_FROM_BOTTOM:
        y = ORIG_SIZE - y
    s = IMG_SIZE / ORIG_SIZE
    return x * s, y * s, r * s

## 3. Image-wise split **before** augmentation (prevents leakage)

Rotations of one source image can never straddle train/test. The test set is **never**
augmented, so lesion coordinates map directly onto test images (angle 0).

In [ ]:
df = df[df['image_id'].apply(lambda i: os.path.exists(os.path.join(IMAGE_DIR, i + '.pgm')))].reset_index(drop=True)
print('Images with files present:', len(df))

train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df['label'], random_state=RANDOM_STATE)
# carve a validation slice out of train (for early stopping)
train_df, val_df = train_test_split(
    train_df, test_size=0.20, stratify=train_df['label'], random_state=RANDOM_STATE)

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name:5s}: {len(d):3d} images  dist={d["label"].value_counts().to_dict()}')

In [ ]:
def build_set(frame, augment):
    X, Y = [], []
    angles = ROT_ANGLES if augment else [0]
    for _, row in frame.iterrows():
        base = load_and_preprocess(row['image_id'])
        if base is None:
            continue
        for a in angles:
            X.append(rotate(base, a))
            Y.append(row['label'])
    return np.array(X, dtype=np.float32), np.array(Y)

x_train, y_train = build_set(train_df, augment=True)
x_val,   y_val   = build_set(val_df,   augment=False)
x_test,  y_test  = build_set(test_df,  augment=False)
print('x_train:', x_train.shape, 'counts', np.bincount(y_train))
print('x_val  :', x_val.shape,   'counts', np.bincount(y_val))
print('x_test :', x_test.shape,  'counts', np.bincount(y_test))

## 4. Attention blocks (corrected)

- **SE** (channel only): squeeze-excite via GAP -> bottleneck -> sigmoid gate.
- **CBAM** (channel + spatial): channel attention uses a **shared MLP over avg- and
  max-pool** (the original 6_15 used avg-pool only); spatial attention convolves over the
  **concat of channel-wise avg and max** maps. The spatial sigmoid map is named `*_spatial`
  so we can read it out for localization.

In [ ]:
def se_block(x, ratio=16):
    f = x.shape[-1]
    s = layers.GlobalAveragePooling2D()(x)
    s = layers.Reshape((1, 1, f))(s)
    s = layers.Dense(max(1, f // ratio), activation='relu', use_bias=False,
                     kernel_initializer='he_normal')(s)
    s = layers.Dense(f, activation='sigmoid', use_bias=False,
                     kernel_initializer='he_normal')(s)
    return layers.Multiply()([x, s])

def cbam_block(x, ratio=16, name='cbam'):
    f = x.shape[-1]
    # ---- channel attention (shared MLP over avg + max pool) ----
    shared1 = layers.Dense(max(1, f // ratio), activation='relu', kernel_initializer='he_normal')
    shared2 = layers.Dense(f, kernel_initializer='he_normal')
    avg = layers.Reshape((1, 1, f))(layers.GlobalAveragePooling2D()(x))
    mx  = layers.Reshape((1, 1, f))(layers.GlobalMaxPooling2D()(x))
    ch  = layers.Activation('sigmoid')(layers.Add()([shared2(shared1(avg)), shared2(shared1(mx))]))
    x   = layers.Multiply()([x, ch])
    # ---- spatial attention (conv over channel-wise avg + max) ----
    avg_p = layers.Lambda(lambda t: tf.reduce_mean(t, axis=-1, keepdims=True))(x)
    max_p = layers.Lambda(lambda t: tf.reduce_max(t, axis=-1, keepdims=True))(x)
    sp = layers.Concatenate(axis=-1)([avg_p, max_p])
    sp = layers.Conv2D(1, 7, padding='same', activation='sigmoid', name=name + '_spatial')(sp)
    return layers.Multiply()([x, sp])

## 5. Build the attention model

EfficientNetB0 is frozen up to `block6a_activation`; the attention block + head are trainable.
The last spatial tensor is tagged `loc_features` (the Grad-CAM target).

In [ ]:
def build_model(attention='cbam', lr=1e-4):
    base = EfficientNetB0(weights='imagenet', include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x = base.get_layer(TAP_LAYER).output
    if attention == 'se':
        x = se_block(x)
    elif attention == 'cbam':
        x = cbam_block(x)
    elif attention == 'se_cbam':
        x = cbam_block(se_block(x))
    x = layers.Activation('linear', name='loc_features')(x)     # Grad-CAM target
    g = layers.GlobalAveragePooling2D(name='gap')(x)
    g = layers.Dropout(0.5)(g)
    out = layers.Dense(1, activation='sigmoid')(g)
    model = Model(base.input, out)
    model.compile(optimizer=Adam(lr), loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return model

model = build_model(ATTENTION)
print('Trainable params:', f'{int(np.sum([tf.size(w).numpy() for w in model.trainable_weights])):,}')
print('Loc-feature layer output:', model.get_layer('loc_features').output.shape)

## 6. Train (class-weighted, early stopping on val AUC)

In [ ]:
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: cw[0], 1: cw[1]}
print('class_weight:', class_weight)

es = EarlyStopping(monitor='val_auc', mode='max', patience=6, restore_best_weights=True)
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight, callbacks=[es], verbose=1)

## 7. Evaluate classification (CNN head) on the held-out test set

In [ ]:
prob = model.predict(x_test, batch_size=BATCH_SIZE, verbose=0).ravel()
pred = (prob >= 0.5).astype(int)

cnn_acc = accuracy_score(y_test, pred)
cnn_auc = roc_auc_score(y_test, prob)
cm = confusion_matrix(y_test, pred); tn, fp, fn, tp = cm.ravel()
cnn_sens = tp / (tp + fn) if (tp + fn) else 0.0
cnn_spec = tn / (tn + fp) if (tn + fp) else 0.0

print(f'CNN  Accuracy   : {cnn_acc:.4f}')
print(f'CNN  ROC-AUC    : {cnn_auc:.4f}')
print(f'CNN  Sensitivity: {cnn_sens:.4f}   Specificity: {cnn_spec:.4f}')
print(classification_report(y_test, pred, target_names=['Normal', 'Abnormal']))
print('Confusion matrix [[TN FP][FN TP]]:\n', cm)

## 8. Hybrid RBF-SVM on **trained** attention features

The feature extractor is cut from the *trained* model at `gap`, so the SVM consumes the
learned attention representation (not a randomly-initialised one).

In [ ]:
feature_extractor = Model(model.input, model.get_layer('gap').output)

def extract(data):
    out = []
    for i in range(0, len(data), BATCH_SIZE):
        out.append(feature_extractor.predict(data[i:i + BATCH_SIZE], verbose=0))
    return np.vstack(out)

f_train = extract(np.concatenate([x_train, x_val], 0))
y_train_full = np.concatenate([y_train, y_val], 0)
f_test  = extract(x_test)

scaler = StandardScaler().fit(f_train)
f_train_s, f_test_s = scaler.transform(f_train), scaler.transform(f_test)

svm = SVC(C=10, gamma='scale', kernel='rbf', probability=True,
          class_weight='balanced', random_state=42).fit(f_train_s, y_train_full)

svm_pred = svm.predict(f_test_s)
svm_prob = svm.predict_proba(f_test_s)[:, 1]
svm_acc = accuracy_score(y_test, svm_pred)
svm_auc = roc_auc_score(y_test, svm_prob)
print(f'SVM  Accuracy: {svm_acc:.4f}   ROC-AUC: {svm_auc:.4f}')
print(classification_report(y_test, svm_pred, target_names=['Normal', 'Abnormal']))

## 9. Weakly-supervised localization: Grad-CAM + CBAM spatial map

`gradcam` works for every attention setting (the fair cross-ablation signal).
`cbam_spatial_map` reads CBAM's native spatial attention when present.

In [ ]:
def gradcam(mdl, arr, layer='loc_features'):
    gm = Model(mdl.inputs, [mdl.get_layer(layer).output, mdl.output])
    with tf.GradientTape() as tape:
        conv, pred = gm(arr)
        loss = pred[:, 0]
    grads = tape.gradient(loss, conv)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heat = tf.reduce_sum(conv[0] * pooled, axis=-1)
    heat = tf.nn.relu(heat)
    heat = heat / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy()

def cbam_spatial_map(mdl, arr):
    names = [l.name for l in mdl.layers if l.name.endswith('_spatial')]
    if not names:
        return None
    sm = Model(mdl.inputs, mdl.get_layer(names[0]).output)
    return sm.predict(arr, verbose=0)[0, :, :, 0]

In [ ]:
def localization_metrics(mdl, frame, use='gradcam', verbose=False):
    hits, dists, ious = [], [], []
    diag = IMG_SIZE * np.sqrt(2)
    for _, row in frame.iterrows():
        if row['label'] != 1 or len(row['lesions']) == 0:
            continue
        base = load_and_preprocess(row['image_id'])
        if base is None:
            continue
        arr = base[None, ...]
        heat = gradcam(mdl, arr) if use == 'gradcam' else cbam_spatial_map(mdl, arr)
        if heat is None:
            return None
        heat = cv2.resize(heat.astype(np.float32), (IMG_SIZE, IMG_SIZE))

        gt = np.zeros((IMG_SIZE, IMG_SIZE), np.uint8)
        centers = []
        for (x, y, r) in row['lesions']:
            cx, cy, rr = scale_lesion(x, y, r)
            centers.append((cx, cy))
            cv2.circle(gt, (int(round(cx)), int(round(cy))), max(1, int(round(rr))), 1, -1)

        py, px = np.unravel_index(int(np.argmax(heat)), heat.shape)
        hits.append(int(gt[py, px] == 1))                                  # pointing game
        dists.append(min(np.hypot(px - cx, py - cy) for cx, cy in centers) / diag)
        pred = (heat >= 0.5 * heat.max()).astype(np.uint8)
        inter = np.logical_and(pred, gt).sum(); union = np.logical_or(pred, gt).sum()
        ious.append(inter / union if union else 0.0)
    if not hits:
        return None
    res = dict(n=len(hits), pointing_game=float(np.mean(hits)),
               centre_dist=float(np.mean(dists)), iou=float(np.mean(ious)))
    if verbose:
        print(f"[{use}] n={res['n']}  pointing-game={res['pointing_game']:.3f}  "
              f"centre-dist={res['centre_dist']:.3f}  IoU={res['iou']:.3f}")
    return res

loc_gradcam = localization_metrics(model, test_df, 'gradcam', verbose=True)
loc_cbam    = localization_metrics(model, test_df, 'cbam',    verbose=True)

### 9b. Qualitative overlays (a few abnormal test cases)

In [ ]:
abn = test_df[(test_df['label'] == 1) & (test_df['lesions'].str.len() > 0)].head(6)
n = len(abn)
if n:
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.2), dpi=120)
    axes = np.atleast_1d(axes)
    for ax, (_, row) in zip(axes, abn.iterrows()):
        base = load_and_preprocess(row['image_id'])
        heat = cv2.resize(gradcam(model, base[None, ...]), (IMG_SIZE, IMG_SIZE))
        ax.imshow(base[..., 0] / 255.0, cmap='gray')
        ax.imshow(heat, cmap='jet', alpha=0.40)
        for (x, y, r) in row['lesions']:
            cx, cy, rr = scale_lesion(x, y, r)
            ax.add_patch(plt.Circle((cx, cy), rr, color='lime', fill=False, lw=1.6))
        ax.set_title(f"{row['image_id']} ({row['severity']})", fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'localization_overlays.png'), dpi=200, bbox_inches='tight')
    plt.show()
    print('Green circle = GT lesion; heatmap = model attention (Grad-CAM).')

## 10. Ablation - does attention help **classification AND localization**?

Retrains `none / se / cbam / se_cbam` and reports test AUC + pointing-game hit-rate.
Slowest cell; reduce `ABL_EPOCHS` for a quick pass.

In [ ]:
ABL_EPOCHS = 15
rows = []
for att in ['none', 'se', 'cbam', 'se_cbam']:
    m = build_model(att)
    m.fit(x_train, y_train, validation_data=(x_val, y_val),
          epochs=ABL_EPOCHS, batch_size=BATCH_SIZE, class_weight=class_weight,
          callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=5,
                                   restore_best_weights=True)], verbose=0)
    p = m.predict(x_test, verbose=0).ravel()
    auc = roc_auc_score(y_test, p)
    lg = localization_metrics(m, test_df, 'gradcam')
    rows.append({'attention': att, 'test_AUC': round(auc, 3),
                 'pointing_game': round(lg['pointing_game'], 3) if lg else None,
                 'centre_dist': round(lg['centre_dist'], 3) if lg else None,
                 'IoU': round(lg['iou'], 3) if lg else None})
    print(rows[-1])

ablation_df = pd.DataFrame(rows)
ablation_df.to_csv(os.path.join(SAVE_DIR, 'attention_ablation.csv'), index=False)
ablation_df

## 11. Paper-ready summary

In [ ]:
n_total = len(df); n_norm = int((df['label'] == 0).sum()); n_abn = int((df['label'] == 1).sum())
lg = loc_gradcam or {'n': 0, 'pointing_game': 0, 'centre_dist': 0, 'iou': 0}

summary = (
    f"On the full Mini-MIAS database (N = {n_total}; {n_norm} normal, {n_abn} abnormal), an "
    f"EfficientNetB0 backbone augmented with a {ATTENTION.upper()} attention block was trained "
    f"to discriminate normal vs. abnormal mammograms using image-level labels only, with an "
    f"image-wise 80/20 split and augmentation restricted to the training set to prevent leakage. "
    f"The CNN head reached {cnn_acc*100:.1f}% accuracy and an ROC-AUC of {cnn_auc:.3f} "
    f"(sensitivity {cnn_sens*100:.1f}%, specificity {cnn_spec*100:.1f}%); an RBF-SVM on the "
    f"trained attention features reached ROC-AUC {svm_auc:.3f}. Crucially, without any "
    f"localization supervision, the model's Grad-CAM attention localized the lesion in "
    f"{lg['pointing_game']*100:.1f}% of {lg['n']} annotated abnormal test images "
    f"(pointing-game), with a mean normalized centre-distance of {lg['centre_dist']:.3f} and "
    f"attention-vs-lesion IoU of {lg['iou']:.3f}, validated against the MIAS ground-truth "
    f"lesion coordinates."
)
print(summary)
with open(os.path.join(SAVE_DIR, 'attention_localization_summary.txt'), 'w') as f:
    f.write(summary)
print('\nSaved -> attention_localization_summary.txt')

## 12. Save model + SVM + scaler

In [ ]:
# save_weights (not .keras): the CBAM Lambda/Normalization layers don't JSON-serialize;
# rebuild the architecture from build_model() then load_weights to restore.
model.save_weights(os.path.join(SAVE_DIR, f'efficientnet_{ATTENTION}_mias.weights.h5'))
joblib.dump(svm,    os.path.join(SAVE_DIR, f'svm_{ATTENTION}_mias.pkl'))
joblib.dump(scaler, os.path.join(SAVE_DIR, f'scaler_{ATTENTION}_mias.pkl'))
print('Saved to', SAVE_DIR)